# Caso de Estudio: Predicción de Resistencia del Hormigón

**1. Contexto del Problema**

En la ingeniería civil, la calidad del hormigón es crítica para la seguridad de edificios y puentes. El método tradicional para verificar su resistencia es lento y costoso:

- Se diseña la mezcla.

- Se crean cilindros de prueba.

- Se espera 28 días de curado.

- Se rompen los cilindros en una prensa hidráulica para medir su resistencia a la compresión (en MPa).

El Problema: Si la mezcla falla a los 28 días, hemos perdido un mes entero de obra y mucho dinero. El Objetivo: Crear un modelo de Machine Learning capaz de predecir la resistencia final del hormigón basándose únicamente en su "receta" (ingredientes y edad), sin tener que esperar a romperlo.

## Dataset

Utilizaremos datos reales donados por el Prof. I-Cheng Yeh (Universidad de Chu-Hua), provenientes de 1030 ensayos de laboratorio.

El Reto Matemático: La química del cemento es compleja y no lineal. La relación entre la cantidad de agua y la resistencia no es una línea recta. Por eso, modelos simples como la Regresión Lineal suelen fallar, mientras que algoritmos basados en árboles (como Random Forest) pueden capturar estas reacciones químicas complejas.

### Variables

**Variables Predictoras ($X$):**

- Cemento (Cement): Kg en un metro cúbico ($m^3$) de mezcla.
- Escoria de Alto Horno (Blast Furnace Slag): Residuo industrial usado como aditivo.
- Ceniza Volante (Fly Ash): Partículas de la quema de carbón, mejora la durabilidad.
- Agua (Water): Componente vital para la hidratación.
- Superplastificante (Superplasticizer): Químico para reducir agua manteniendo fluidez.
- Agregado Grueso (Coarse Aggregate): Grava/Piedras.
- Agregado Fino (Fine Aggregate): Arena.
- Edad (Age): Días de curado (de 1 a 365 días).

**Variable Objetivo ($y$):**

- Resistencia a la Compresión (Concrete compressive strength): Medida en MegaPascales (MPa). Es lo que queremos predecir.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# !pip install xgboost
from xgboost import XGBRegressor
# !pip install lightgbm
from lightgbm import LGBMRegressor

In [2]:
#from google.colab import files
#files.upload()

#data = pd.read_csv('Country-data.csv')

In [3]:
df = pd.read_csv('data/concrete.csv')
df.head()

,cement,slag,ash,water,superplastic,coarseagg,fineagg,age,strength
0,141.3,212.0,0.0,203.5,0.0,971.8,748.5,28,29.89
1,168.9,42.2,124.3,158.3,10.8,1080.8,796.2,14,23.51
2,250.0,0.0,95.7,187.4,5.5,956.9,861.2,28,29.22
3,266.0,114.0,0.0,228.0,0.0,932.0,670.0,28,45.85
4,154.8,183.4,0.0,193.3,9.1,1047.4,696.7,28,18.29


In [4]:
df.describe()

,cement,slag,ash,water,superplastic,coarseagg,fineagg,age,strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [5]:
df.shape

(1030, 9)

In [6]:
# 1. Ver los nombres originales
print("Columnas originales:", df.columns.tolist())

# 2. Renombrar columnas en español
df.columns = ['Cemento', 'Escoria', 'Ceniza', 'Agua', 'Superplast', 'Grano_Grueso', 'Grano_Fino', 'Edad', 'Resistencia']

df.head()

Columnas originales: ['cement', 'slag', 'ash', 'water', 'superplastic', 'coarseagg', 'fineagg', 'age', 'strength']


,Cemento,Escoria,Ceniza,Agua,Superplast,Grano_Grueso,Grano_Fino,Edad,Resistencia
0,141.3,212.0,0.0,203.5,0.0,971.8,748.5,28,29.89
1,168.9,42.2,124.3,158.3,10.8,1080.8,796.2,14,23.51
2,250.0,0.0,95.7,187.4,5.5,956.9,861.2,28,29.22
3,266.0,114.0,0.0,228.0,0.0,932.0,670.0,28,45.85
4,154.8,183.4,0.0,193.3,9.1,1047.4,696.7,28,18.29
